# CIRF-TOPSIS: Xining baseline (13 objects)

Source: https://doi.org/10.1007/s10489-025-07049-1

Adaptation: Uses a label-free graph and rough-set adaptation followed by TOPSIS. No labeled GA or XGBoost stage is run.

Settings: PC level=0.001; 100 bootstrap samples; retain edges in at least 70; direct-information share=0.1; seed=2026.

The small input arrays are embedded below. See README.md for shared settings, execution, and interpretation. Parameters were selected on these same objects using CAS.


In [ ]:
import hashlib
import json
from itertools import combinations

import numpy as np

# Fixed 13-object input snapshot; no external notebook is required.
values = {'X1': [[232844400, 363040200, 951871500, 16028100, 12839400, 63332100],
        [498796200, 836140500, 1523463300, 7876800, 77425200, 145968300],
        [590792400, 408241800, 1264804200, 8746200, 119119500, 27847800],
        [453535200, 281313000, 2052270900, 45549900, 29835000, 26512200],
        [109103400, 84975300, 478243800, 2700900, 22102200, 0],
        [43988400, 3285900, 40133700, 1599300, 48115800, 0],
        [219577500, 331435800, 791136000, 0, 28975500, 45741600],
        [1898100, 186300, 13910400, 53100, 9067500, 0], [6731100, 1264500, 37102500, 827100, 23383800, 0],
        [10149300, 624600, 69009300, 2301300, 32026500, 0],
        [783862200, 172843200, 847346400, 14004000, 24583500, 8267400],
        [480004200, 385567200, 1648658700, 9315900, 40091400, 17743500],
        [638441100, 723882600, 1665636300, 16281900, 81209700, 86106600]],
 'X2': [[234132300, 364105800, 949576500, 15961500, 12839400, 63340200],
        [499432500, 838892700, 1520082900, 7876800, 77425200, 145960200],
        [590792400, 408241800, 1264804200, 8746200, 119119500, 27847800],
        [456410700, 281717100, 2048993100, 45548100, 29835000, 26512200],
        [109103400, 84975300, 478243800, 2700900, 22102200, 0],
        [43988400, 3285900, 40133700, 1599300, 48115800, 0],
        [219577500, 331435800, 791136000, 0, 28975500, 45741600],
        [1898100, 186300, 13910400, 53100, 9067500, 0], [6731100, 1264500, 37102500, 827100, 23383800, 0],
        [10149300, 624600, 69009300, 2301300, 32026500, 0],
        [783862200, 172845900, 847343700, 14004000, 24583500, 8267400],
        [480004200, 385567200, 1648658700, 9315900, 40091400, 17743500],
        [638441100, 723885300, 1665633600, 16281900, 81209700, 86106600]],
 'X3': [[234090900, 363079800, 950717700, 15875100, 12839400, 63352800],
        [499121100, 836076600, 1523223000, 7876800, 77425200, 145947600],
        [590792400, 408241800, 1264804200, 8746200, 119119500, 27847800],
        [456381000, 281336400, 2049408900, 45542700, 29835000, 26512200],
        [109103400, 84975300, 478243800, 2700900, 22102200, 0],
        [43988400, 3285900, 40133700, 1599300, 48115800, 0],
        [219577500, 331435800, 791136000, 0, 28975500, 45741600],
        [1898100, 186300, 13910400, 53100, 9067500, 0], [6731100, 1264500, 37102500, 827100, 23383800, 0],
        [10149300, 624600, 69009300, 2301300, 32026500, 0],
        [783862200, 172844100, 847345500, 14004000, 24583500, 8267400],
        [480004200, 385567200, 1648658700, 9315900, 40091400, 17743500],
        [638441100, 723882600, 1665636300, 16281900, 81209700, 86106600]],
 'X4': [[233073000, 364482000, 950026500, 16191000, 12839400, 63343800],
        [497124900, 842259600, 1519027200, 7876800, 77425200, 145956600],
        [590792400, 408241800, 1264804200, 8746200, 119119500, 27847800],
        [453763800, 281916900, 2051433900, 45554400, 29835000, 26512200],
        [109103400, 84975300, 478243800, 2700900, 22102200, 0],
        [43988400, 3285900, 40133700, 1599300, 48115800, 0],
        [219577500, 331435800, 791136000, 0, 28975500, 45741600],
        [1898100, 186300, 13910400, 53100, 9067500, 0], [6731100, 1264500, 37102500, 827100, 23383800, 0],
        [10149300, 624600, 69009300, 2301300, 32026500, 0],
        [783862200, 172849500, 847340100, 14004000, 24583500, 8267400],
        [480004200, 385567200, 1648658700, 9315900, 40091400, 17743500],
        [638441100, 723893400, 1665625500, 16281900, 81209700, 86106600]],
 'X_GNN': [[1.2847, 0.0328, 0.9199, 0.547, 0.9623, 1.0416], [1.4388, 0.2173, 0.9251, 0.3661, 0.7657, 1.0301],
           [0.5799, 0.4036, 0.5127, 0.3826, 0.3668, 0.6181], [0.6514, 0.0, 0.6561, 0.5955, 0.7861, 0.7097],
           [1.0569, 0.2446, 1.1748, 0.8674, 0.9807, 1.2204], [0.3136, 0.3107, 0.2432, 0.2576, 0.2307, 0.356],
           [1.4631, 0.2302, 1.0372, 0.4263, 0.813, 1.0689], [0.1929, 0.1989, 0.1862, 0.1943, 0.1937, 0.1882],
           [0.3136, 0.3107, 0.2432, 0.2576, 0.2307, 0.356], [0.3136, 0.3107, 0.2432, 0.2576, 0.2307, 0.356],
           [0.39, 0.2622, 0.5323, 0.5137, 0.4475, 0.623], [1.2794, 0.2737, 1.2803, 0.8464, 1.0261, 1.3336],
           [1.5277, 0.3486, 1.2552, 0.6476, 0.9244, 1.339]],
 'w_e': [0.2506, 0.2519, 0.2607, 0.2368],
 'C_chengben': [5]}
DATA_SHA256 = hashlib.sha256(
    json.dumps(values, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()

raw = np.stack([np.asarray(values["X" + str(i)], dtype=float) for i in range(1, 5)])
w_E = np.asarray(values["w_e"], dtype=float)
w_E = w_E / w_E.sum()
cost_idx = tuple(int(j) for j in values["C_chengben"])

# Identical external attribute weights for all six updated baselines.
# The proposed model retains its own attribute-weighting calculation.
w_A = np.asarray([0.247, 0.088, 0.259, 0.166, 0.192, 0.048], dtype=float)
q = 5  # Shared q-rung setting.

assert raw.shape == (4, 13, 6), raw.shape
assert w_E.shape == (4,) and np.isclose(w_E.sum(), 1)
assert w_A.shape == (6,) and abs(w_A.sum() - 1.001) < 1e-12
assert cost_idx == (5,)

# Shared GNN output is the primary 13 x 6 input to all six baselines.
X_GNN = np.asarray(values["X_GNN"], dtype=float)
assert X_GNN.shape == (13, 6)

def FuzzyNorm(X, q, C_chengben, eps=0.05):
    """Convert rows to q-ROF pairs with deterministic boundary treatment."""
    if q < 1 or not 0 < eps < 1:
        raise ValueError("Require q >= 1 and 0 < eps < 1")

    Min = []
    Max = []
    for i in range(len(X)):
        Min.append(min(X[i]))
        Max.append(max(X[i]))

    U = [[] for _ in range(len(X))]
    V = [[] for _ in range(len(X))]
    for i in range(len(X)):
        if Max[i] == Min[i]:
            raise ValueError(f"Object {i+1} has constant attributes; row normalization is undefined")
        for j in range(len(X[0])):
            if j not in C_chengben:
                u = round((X[i][j] - Min[i]) / (Max[i] - Min[i]), 3)
                v = round((1 - u**q)**(1/q), 3)
            else:
                v = round((Max[i] - X[i][j]) / (Max[i] - Min[i]), 3)
                u = round((1 - v**q)**(1/q), 3)
            # Independent three-decimal rounding can cross the q-ROF boundary.
            # Keep the normalized component and lower only the derived one.
            if u**q + v**q > 1 + 1e-12:
                if j not in C_chengben:
                    v = int((max(0.0, 1-u**q)**(1/q))*1000 + 1e-12) / 1000
                else:
                    u = int((max(0.0, 1-v**q)**(1/q))*1000 + 1e-12) / 1000
            U[i].append(u)
            V[i].append(v)

    high = (1 - eps**q)**(1/q)
    if high >= 1.0:
        raise ValueError("The chosen q and eps round high to 1; increase eps")

    for i in range(len(U)):
        for j in range(len(U[0])):
            if U[i][j] == 0.0 and V[i][j] == 1.0:
                U[i][j] = eps
                V[i][j] = high
            elif U[i][j] == 1.0 and V[i][j] == 0.0:
                U[i][j] = high
                V[i][j] = eps

    return U, V

# Retain the four original expert matrices for methods that need expert
# disagreement or expert-level graph/rule structure. Their weights are fixed.
U_GNN, V_GNN = FuzzyNorm(X_GNN, q, cost_idx, eps=0.05)
X = np.asarray(U_GNN, dtype=float)
X_nonmembership = np.asarray(V_GNN, dtype=float)
expert_pairs = [FuzzyNorm(raw[e], q, cost_idx, eps=0.05) for e in range(raw.shape[0])]
utility = np.asarray([pair[0] for pair in expert_pairs], dtype=float)
expert_nonmembership = np.asarray([pair[1] for pair in expert_pairs], dtype=float)
assert utility.shape == (4, 13, 6) and X.shape == (13, 6)
assert np.isfinite(utility).all() and np.isfinite(X).all()
assert np.all((utility >= 0) & (utility <= 1))
assert np.all((X >= 0) & (X <= 1))
assert np.array_equal(X[5], X[8]) and np.array_equal(X[5], X[9])
assert np.array_equal(X_nonmembership[5], X_nonmembership[8]) and np.array_equal(X_nonmembership[5], X_nonmembership[9])

def report_ranking(scores, decimals=12):
    scores = np.asarray(scores, dtype=float)
    if scores.shape != (13,) or not np.isfinite(scores).all():
        raise ValueError("Expected 13 finite scores, got " + repr(scores))
    order = np.argsort(-scores, kind="stable")
    groups = []
    for idx in order:
        if groups and np.isclose(scores[idx], scores[groups[-1][0]], rtol=1e-12, atol=1e-15):
            groups[-1].append(int(idx))
        else:
            groups.append([int(idx)])
    expression = " > ".join(" = ".join("O" + str(i + 1) for i in group) for group in groups)
    print("Final ranking (= denotes a tie):")
    print(expression)
    print("\nScores by object:")
    for i, s in enumerate(scores):
        print("O{:02d}: {:.{}g}".format(i + 1, s, decimals))
    print("\nEmbedded data SHA256:", DATA_SHA256)
    return groups

print("Input: 4 experts, 13 objects, 6 attributes; attribute 6 is cost-type")
print("Expert weights:", w_E.tolist())
print("Shared attribute weights:", w_A.tolist())
print("q =", q)
print("Shared input: X_GNN -> FuzzyNorm(q=5, eps=0.05)")


Input: 4 experts, 13 objects, 6 attributes; attribute 6 is cost-type
Expert weights: [0.25060000000000004, 0.25190000000000007, 0.26070000000000004, 0.23680000000000004]
Shared attribute weights: [0.285, 0.066, 0.301, 0.137, 0.152, 0.06]
q = 5
Shared input: X_GNN -> FuzzyNorm(q=5, eps=0.05)


## Method computation


In [2]:
from math import erfc, sqrt

# Paper p.6: remove one of any pair with |Pearson r| > 0.95.
keep = []
for j in range(X.shape[1]):
    if all(abs(np.corrcoef(X[:, j], X[:, k])[0, 1]) <= .95 for k in keep):
        keep.append(j)
Y = X[:, keep]
wc = w_A[keep] / w_A[keep].sum()

def ci_pvalue(A, i, j, cond):
    cols = [i, j] + list(cond)
    Z = A[:, cols]
    if np.any(Z.std(axis=0) < 1e-12):
        return 1.0
    C = np.corrcoef(Z, rowvar=False)
    P = np.linalg.pinv(C)
    r = -P[0, 1] / np.sqrt(max(P[0, 0] * P[1, 1], 1e-24))
    r = np.clip(r, -0.999999, 0.999999)
    dof = A.shape[0] - len(cond) - 3
    if dof <= 0:
        return 0.0
    return float(erfc(abs(np.arctanh(r) * np.sqrt(dof)) / sqrt(2)))

def pc_skeleton(A, alpha=0.001, max_cond=4):
    p = A.shape[1]
    adj = np.ones((p, p), dtype=bool)
    np.fill_diagonal(adj, False)
    sepset = {}
    for depth in range(min(max_cond, p-2) + 1):
        frozen = adj.copy()
        for i in range(p):
            for j in range(i+1, p):
                if not frozen[i, j]:
                    continue
                candidates = set()
                for root, other in [(i,j), (j,i)]:
                    neighbors = [k for k in range(p) if frozen[root,k] and k != other]
                    candidates.update(combinations(neighbors, depth))
                for cond in sorted(candidates):
                    if ci_pvalue(A, i, j, cond) > alpha:
                        adj[i,j] = adj[j,i] = False
                        sepset[(i,j)] = set(cond)
                        break
    return adj, sepset

full_adj, sep = pc_skeleton(Y)
# Collider orientation where PC identifies it; unresolved skeleton edges stay bidirectional.
direction = full_adj.copy()
for middle in range(Y.shape[1]):
    neighbors = np.flatnonzero(full_adj[:, middle])
    for left, right in combinations(neighbors, 2):
        if not full_adj[left, right] and middle not in sep.get(tuple(sorted((left,right))), set()):
            direction[middle, left] = False
            direction[middle, right] = False

rng = np.random.RandomState(2026)
count = np.zeros_like(full_adj, dtype=int)
for _ in range(100):
    sample = Y[rng.choice(len(Y), size=len(Y), replace=True)]
    sampled_adj, _ = pc_skeleton(sample)
    count += sampled_adj.astype(int)
stable = count >= 70
direction &= stable
for i in range(Y.shape[1]):
    for j in range(i+1, Y.shape[1]):
        if stable[i,j] and not (direction[i,j] or direction[j,i]):
            direction[i,j] = direction[j,i] = True

forward_upper = np.empty_like(Y); forward_lower = np.empty_like(Y)
backward_upper = np.empty_like(Y); backward_lower = np.empty_like(Y)
for j in range(Y.shape[1]):
    outgoing = np.flatnonzero(direction[j])
    incoming = np.flatnonzero(direction[:,j])
    if len(outgoing) == 0: outgoing = np.array([j])
    if len(incoming) == 0: incoming = np.array([j])
    forward_upper[:,j] = Y[:,outgoing].max(axis=1)
    forward_lower[:,j] = Y[:,outgoing].min(axis=1)
    backward_upper[:,j] = Y[:,incoming].max(axis=1)
    backward_lower[:,j] = Y[:,incoming].min(axis=1)

# Label-free replacement of the paper's supervised GA choice among five systems.
S_final = 0.1*Y + 0.225*(forward_upper + forward_lower + backward_upper + backward_lower)
norms = np.linalg.norm(S_final, axis=0)
weighted = (S_final / np.maximum(norms, 1e-15)) * wc
ideal_best = weighted.max(axis=0)
ideal_worst = weighted.min(axis=0)
d_best = np.linalg.norm(weighted - ideal_best, axis=1)
d_worst = np.linalg.norm(weighted - ideal_worst, axis=1)
scores = d_worst / np.maximum(d_best + d_worst, 1e-15)
print("Retained attribute columns (1-based):", [k+1 for k in keep])
print("Stable PC edges:", int(np.triu(stable,1).sum()), "/", int(Y.shape[1]*(Y.shape[1]-1)/2))
print("Five equally weighted information systems + TOPSIS; no labeled GA/XGBoost stage")


Retained attribute columns (1-based): [1, 2, 3, 4, 5, 6]
Stable PC edges: 1 / 15
Five equally weighted information systems + TOPSIS; no labeled GA/XGBoost stage


## Ranking and scores


In [3]:
rank_groups = report_ranking(scores)


Final ranking (= denotes a tie):
O8 > O6 = O9 = O10 > O4 > O12 > O5 > O1 > O13 > O7 > O2 > O11 > O3

Scores by object:
O01: 0.340256063854
O02: 0.277219758253
O03: 0.226308639062
O04: 0.390691715493
O05: 0.352045484216
O06: 0.478691516687
O07: 0.284753970627
O08: 0.751265150914
O09: 0.478691516687
O10: 0.478691516687
O11: 0.268691588202
O12: 0.352229003193
O13: 0.297693884733

Embedded data SHA256: 915a03f9bef24a3f651d3398d3a590e28027949269ab027c0c1096aef4db92a5
